# Model performance evaluation

### Imports

In [ ]:
import torch
from torch_geometric.loader import DataLoader

from gnn_ai_code_detector.train      import ModelCheckpoint, predict
from gnn_ai_code_detector.split      import get_huvsai_split
from gnn_ai_code_detector.model      import CCppGNN
from gnn_ai_code_detector.dataset    import CCppDataset
from gnn_ai_code_detector.preprocess import CCppPreprocessor

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

import random
import numpy as np
import pandas as pd
from pathlib import Path

### Random state initialization

In [3]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE), np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_STATE)

### Model loading

In [4]:
languages = [("C", ), ("C++", ), ("C", "C++")]

checkpoints = {
    langs: ModelCheckpoint.load(Path(f"../../models/{"_".join(langs)}_best_model.pt"))
    for langs in languages
}

if torch.xpu.is_available():
    device = "xpu"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

models = {
    langs: CCppGNN(
        check.vocab,
        check.embedding_dims,
        check.num_relations,
        check.bool_features,
    ).to(device)
    for langs, check in checkpoints.items()
}

for langs, model in models.items():
    model.load_state_dict(checkpoints[langs].state_dict)
    model.eval()

### Data reconstruction

In [5]:
df = pd.read_csv("../../data/Code_Dataset/HumanVsAi_CodeDataset.csv")

def get_test_indices(languages: list[str]) -> list[int]:
    _, __test_indices = get_huvsai_split(
        df, Path("../../data/c_cpp/clean_asts"), languages,
        test_size=0.4, random_state=RANDOM_STATE
    )

    test_indices, _ = get_huvsai_split(
        df.loc[__test_indices],
        Path("../../data/c_cpp/clean_asts"), languages,
        test_size=0.5, random_state=RANDOM_STATE
    )

    return test_indices

preprocessor = CCppPreprocessor("") # Clang is never invoked

clean_asts = Path("../../data/c_cpp/clean_asts")

datasets = {
    langs: CCppDataset(
        get_test_indices(list(langs)),
        clean_asts, df, preprocessor
    ) for langs in languages
}

for langs, dataset in datasets.items():
    dataset.vocab = checkpoints[langs].vocab

loaders = {
    langs: DataLoader(dataset, batch_size=32, shuffle=False)
    for langs, dataset in datasets.items()
}

### Metrics calculation

In [ ]:
def calculate_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":          accuracy_score(y_true, y_pred),
        "precision":         precision_score(y_true, y_pred),
        "recall":            recall_score(y_true, y_pred),
        "f1":                f1_score(y_true, y_pred),
        "auroc":             roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob)
    }

results = {}

for langs in languages:
    y_true, y_pred, y_prob = predict(
        models[langs],
        loaders[langs],
        device,
    )

    results[langs] = {
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

metrics = {
    langs: calculate_metrics(**result)
    for langs, result in results.items()
}

metrics_df = pd.DataFrame(
    list(metrics.values()),
    index=["/".join(langs) for langs in languages],
)

print(metrics_df.round(3))

       accuracy  precision  recall     f1  auroc  average_precision
C         0.960      0.979   0.941  0.960  0.994              0.994
C++       0.989      0.986   0.990  0.988  0.999              0.999
C/C++     0.973      0.953   0.988  0.970  0.998              0.997


In [35]:
def print_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm = cm / cm.sum(axis=1, keepdims=True)
    print( "       Human   AI")
    print(f"Human  {cm[0,0]:.3f}   {cm[0,1]:.3f}")
    print(f"AI     {cm[1,0]:.3f}   {cm[1,1]:.3f}")

for langs, result in results.items():
    print(f"\n{'/'.join(langs)}")
    print_confusion_matrix(result["y_true"], result["y_pred"])


C
       Human   AI
Human  0.980   0.020
AI     0.059   0.941

C++
       Human   AI
Human  0.988   0.012
AI     0.010   0.990

C/C++
       Human   AI
Human  0.961   0.039
AI     0.012   0.988
